# 00 — Data Profile

**Purpose:** Inspect shape, completeness, distributions, and correlations across all cleaned datasets before any analysis.

**Datasets covered:**
- `national_enriched.csv` — 11 fiscal years × 20 national indicators
- `county_enriched.csv` — 517 county × FY rows × 12 columns
- `composite_scores.csv` — service delivery scores (where indicator data exists)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

# Load cleaned data
DATA = os.path.join("..", "data", "clean")
national = pd.read_csv(os.path.join(DATA, "national_enriched.csv"))
county   = pd.read_csv(os.path.join(DATA, "county_enriched.csv"))
composite = pd.read_csv(os.path.join(DATA, "composite_scores.csv"))

print(f"national_enriched : {national.shape}")
print(f"county_enriched   : {county.shape}")
print(f"composite_scores  : {composite.shape}")

## 1. National Dataset — Shape & Types

In [ ]:
print("Columns:", national.columns.tolist())
print("\nData types:")
print(national.dtypes)
print(f"\nFiscal years covered: {national['fiscal_year'].min()} → {national['fiscal_year'].max()}")
national.head()

## 2. Missing Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# National missing %
nat_missing = (national.drop(columns=["fiscal_year"]).isna().mean() * 100).sort_values(ascending=False)
nat_missing.plot.barh(ax=axes[0], color="salmon")
axes[0].set_title("National: % Missing by Column")
axes[0].set_xlabel("% Missing")

# County missing %
cty_missing = (county.drop(columns=["fiscal_year", "county_code", "county_name"]).isna().mean() * 100).sort_values(ascending=False)
cty_missing.plot.barh(ax=axes[1], color="steelblue")
axes[1].set_title("County: % Missing by Column")
axes[1].set_xlabel("% Missing")

plt.tight_layout()
plt.show()

print("\n--- National missing values ---")
print(nat_missing.to_string())
print(f"\n--- County missing values ---")
print(cty_missing.to_string())

## 3. Summary Statistics

In [ ]:
print("=== NATIONAL ===")
display(national.describe().T)

print("\n=== COUNTY ===")
display(county.describe().T)

## 4. Distribution of Key Numeric Columns

In [ ]:
# County-level distributions (where we have data)
dist_cols = ["allocation_mn", "expenditure_mn", "absorption_rate_pct"]
available = [c for c in dist_cols if c in county.columns]

fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 4))
if len(available) == 1:
    axes = [axes]
for ax, col in zip(axes, available):
    county[col].dropna().hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(f"County: {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")
plt.suptitle("County-Level Distributions (all FYs pooled)", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

# Indicator distributions (where available)
ind_cols = [c for c in county.columns if c.startswith("ind_")]
if ind_cols:
    fig, axes = plt.subplots(1, len(ind_cols), figsize=(5 * len(ind_cols), 4))
    if len(ind_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, ind_cols):
        county[col].dropna().hist(bins=20, ax=ax, color="darkorange", edgecolor="white")
        ax.set_title(col.replace("ind_", "").replace("_", " ").title())
    plt.suptitle("County Indicator Distributions", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

## 5. Correlation Heatmap — National Indicators

In [ ]:
num_cols = national.select_dtypes(include="number").columns.tolist()
corr = national[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title("National Indicators — Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

## 6. County Data Coverage

How many counties have indicator data across fiscal years?

In [ ]:
# Unique counties and FYs
n_counties = county["county_name"].nunique()
n_fys = county["fiscal_year"].nunique()
print(f"Counties: {n_counties}  |  Fiscal years: {n_fys}  |  Expected rows: {n_counties * n_fys}")
print(f"Actual rows:  {len(county)}")

# Indicator coverage matrix: counties with non-null indicators
ind_cols = [c for c in county.columns if c.startswith("ind_")]
if ind_cols:
    coverage = county.groupby("county_name")[ind_cols].apply(lambda g: g.notna().any()).sum()
    print(f"\nCounties with at least one non-null value per indicator:")
    print(coverage.to_string())

# Composite score coverage
if len(composite) > 0:
    counties_with_score = composite["service_delivery_score"].notna().sum()
    print(f"\nComposite scores computed: {counties_with_score} / {len(composite)} county-FY rows")
else:
    print("\nNo composite scores yet.")

## 7. Key Observations

*Run cells above, then fill in observations:*

- **National data** is complete across 11 FYs (2013/14 – 2023/24) for revenue, budget, and debt.
- **WB indicators** are placeholder (empty values) — need live API fetch or manual entry.
- **County allocations & expenditure** cover all 47 counties × 11 FYs.
- **County indicators** (KDHS/KIHBS) are sparse — only 13 counties with snapshots for 2014 and 2022.
- **Composite scores** are limited to counties with indicator data.
- **Next steps:** Prioritise filling WB data; consider interpolation for missing county indicator years.